In [1]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features

# ---------- 1. Muat dan bersihkan data ----------
# Membaca data polutan NO2
df = pd.read_csv('polutan_gresik_2025_2026.csv')
print(df.columns) # <--- Tambahkan baris ini untuk mengecek
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)

target_pollutant = 'NO2'

# Memastikan kolom target bertipe numerik, error menjadi NaN
df[target_pollutant] = pd.to_numeric(df[target_pollutant], errors='coerce')

n_missing_before = df[target_pollutant].isna().sum()
print(f"Jumlah nilai non-numerik/kosong awal yang dikonversi jadi NaN: {n_missing_before}")


Index(['time', 'NO2', 'CO', 'O3'], dtype='object')
Jumlah nilai non-numerik/kosong awal yang dikonversi jadi NaN: 177


In [2]:
# ---------- 2. Deteksi dan Penghapusan Outliers (Pencilan) ----------
# Menghitung Kuartil 1 (Q1) dan Kuartil 3 (Q3)
Q1 = df[target_pollutant].quantile(0.25)
Q3 = df[target_pollutant].quantile(0.75)

# Menghitung Interquartile Range (IQR)
IQR = Q3 - Q1

# Menentukan batas kewajaran data
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Batas Bawah IQR: {lower_bound:.2f} | Batas Atas IQR: {upper_bound:.2f}")

# Menghapus nilai yang melanggar batas (diubah menjadi NaN)
outliers_condition = (df[target_pollutant] < lower_bound) | (df[target_pollutant] > upper_bound)
df.loc[outliers_condition, target_pollutant] = np.nan

print(f"Jumlah outliers yang terdeteksi dan dikosongkan: {outliers_condition.sum()}")


Batas Bawah IQR: -0.00 | Batas Atas IQR: 0.00
Jumlah outliers yang terdeteksi dan dikosongkan: 9


In [3]:
# ---------- 3. Imputasi Missing Value ----------
# Jadikan kolom tanggal sebagai index sementara untuk interpolasi
df_clean = df.set_index('time')

# Melakukan interpolasi berbasis waktu
df_clean = df_clean.interpolate(method='time')

# Menambal celah di awal atau akhir data jika interpolasi tidak menjangkau
df_clean = df_clean.ffill().bfill()

print(f"Sisa missing value setelah proses imputasi: {df_clean[target_pollutant].isna().sum()}")


Sisa missing value setelah proses imputasi: 0


In [4]:
# ---------- 4. Persiapan Ekstraksi Fitur TSFEL ----------
# Ekstrak data deret waktu sempurna ke dalam bentuk Array 1D
fs = 1
signal_1d = df_clean[target_pollutant].astype(float).values

# Daftar PERSIS fitur yang diminta (68 Fitur)
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print("Jumlah fitur yang disiapkan untuk diekstrak:", len(FEATURE_LIST))


Jumlah fitur yang disiapkan untuk diekstrak: 68


In [5]:
# ---------- 5. Proses Ekstraksi Fitur ----------
def to_scalar(result):
    # Mengubah hasil dictionary dari TSFEL menjadi nilai mentah
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    # Jika hasil berupa array atau list panjang, kita ambil rata-ratanya
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    # Jika sudah skalar, ubah jadi float standar
    return float(result)

def extract_one(fn_name, signal, fs):
    # Memanggil fungsi TSFEL secara dinamis berdasarkan nama fiturnya
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    # Memeriksa apakah fungsi tersebut butuh parameter 'fs'
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)

# Menjalankan iterasi untuk setiap fitur pada sinyal NO2
row = {}
for fn_name in FEATURE_LIST:
    row[fn_name] = extract_one(fn_name, signal_1d, fs)

extracted_features_final = pd.DataFrame([row])

print(f"Berhasil! Ekstraksi menghasilkan {extracted_features_final.shape[1]} fitur.")

# Menyimpan hasil ke dalam format CSV
output_filename = f'{target_pollutant}_Kerek_TSFEL.csv'
extracted_features_final.to_csv(output_filename, index=False)
print(f"File berhasil disimpan sebagai: {output_filename}")


Berhasil! Ekstraksi menghasilkan 68 fitur.
File berhasil disimpan sebagai: NO2_Kerek_TSFEL.csv


In [6]:
# ---------- Ekspor Data Bersih ke test.csv ----------
# Karena variabel 'df_clean' sudah berisi data yang melewati proses 
# deteksi outliers (IQR) dan imputasi missing value (time interpolation),
# kita tinggal menyimpan DataFrame tersebut ke dalam format CSV.

# Reset index (agar kolom 'time' kembali menjadi kolom biasa, bukan index)
# Jika 'time' tidak di-reset, kolom tersebut tidak akan masuk ke dalam CSV.
if df_clean.index.name == 'time':
    df_clean_to_save = df_clean.reset_index()
else:
    df_clean_to_save = df_clean

# Menyimpan ke file test.csv tanpa menyertakan nomor index pandas
df_clean_to_save.to_csv('test.csv', index=False)

print("Berhasil! Data yang sudah dibersihkan (outliers & missing values) telah disimpan ke dalam 'test.csv'.")

Berhasil! Data yang sudah dibersihkan (outliers & missing values) telah disimpan ke dalam 'test.csv'.
